In [2]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# NB_Silver_Orders
# Bronze -> Silver
# ============================================================

SOURCE_TABLE = "bronze_order"
CUSTOMERS_TABLE = "silver_customers"
SILVER_TABLE = "silver_orders"
QUARANTINE_TABLE = "quarantine_orders"

# ------------------------------------------------------------
# 1. Read Bronze
# ------------------------------------------------------------

orders = spark.table(SOURCE_TABLE)

customers = (
    spark.table(CUSTOMERS_TABLE)
    .select("customer_id")
    .dropDuplicates()
)

# ------------------------------------------------------------
# 2. Standardize types
# ------------------------------------------------------------

orders = (
    orders
    .withColumn("order_id", F.col("order_id").cast("int"))
    .withColumn("customer_id", F.col("customer_id").cast("int"))
    .withColumn("order_date", F.to_timestamp("order_date"))
    .withColumn("status", F.lower(F.trim(F.col("status").cast("string"))))
    .withColumn("total_amount", F.col("total_amount").cast("decimal(12,2)"))
)

# ------------------------------------------------------------
# 3. Referential integrity
# ------------------------------------------------------------

orders = (
    orders
    .join(
        customers.withColumn("customer_exists", F.lit(True)),
        on="customer_id",
        how="left"
    )
)

# ------------------------------------------------------------
# 4. Validation
# ------------------------------------------------------------

df_checked = (
    orders
    .withColumn(
        "dq_reason",
        F.when(
            F.col("order_id").isNull(),
            F.lit("NULL_ORDER_ID")
        )
        .when(
            F.col("customer_id").isNull(),
            F.lit("NULL_CUSTOMER_ID")
        )
        .when(
            F.col("customer_exists").isNull(),
            F.lit("INVALID_CUSTOMER_REFERENCE")
        )
        .when(
            F.col("order_date").isNull(),
            F.lit("NULL_ORDER_DATE")
        )
        .when(
            F.col("status").isNull() |
            (F.col("status") == ""),
            F.lit("NULL_OR_EMPTY_STATUS")
        )
        .when(
            F.col("total_amount").isNull(),
            F.lit("NULL_TOTAL_AMOUNT")
        )
        .when(
            F.col("total_amount") < 0,
            F.lit("NEGATIVE_TOTAL_AMOUNT")
        )
        .otherwise(F.lit(None))
    )
)

# ------------------------------------------------------------
# 5. Quarantine
# ------------------------------------------------------------

df_quarantine = (
    df_checked
    .filter(F.col("dq_reason").isNotNull())
    .drop("customer_exists")
    .withColumn("quarantined_at", F.current_timestamp())
)

df_quarantine.write.mode("overwrite").format("delta").saveAsTable(
    QUARANTINE_TABLE
)

# ------------------------------------------------------------
# 6. Valid records
# ------------------------------------------------------------

df_valid = (
    df_checked
    .filter(F.col("dq_reason").isNull())
    .drop("dq_reason", "customer_exists")
)

# ------------------------------------------------------------
# 7. Deduplicate
# ------------------------------------------------------------

window = Window.partitionBy("order_id").orderBy(
    F.col("order_date").desc_nulls_last()
)

df_dedup = (
    df_valid
    .withColumn("row_number", F.row_number().over(window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

# ------------------------------------------------------------
# 8. Write Silver
# ------------------------------------------------------------

df_dedup.write.mode("overwrite").format("delta").saveAsTable(
    SILVER_TABLE
)

print(f"Silver table created: {SILVER_TABLE}")
print(f"Valid records: {df_dedup.count()}")
print(f"Quarantined records: {df_quarantine.count()}")

StatementMeta(, 6e457bf9-245a-4fcb-af9e-714bff0e8ec4, 4, Finished, Available, Finished, False)

Silver table created: silver_orders
Valid records: 49980
Quarantined records: 19
